# Flight Trajectory Prediction Results Analysis

This notebook evaluates the performance of the generative flight trajectory prediction model through various metrics, visualizations, and calibration analysis.

## Setup and Configuration

In [1]:
# Core imports
import sys
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import pathlib

# Add project root to Python path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# External libraries
from traffic.core import Traffic
import pyproj
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Project modules
from utils.utils import cache_paths
from utils.inference_utils import sample_many, denorm_seq_to_global
from utils.metrics import pit_values
from model import load_model_checkpoint as load_base_checkpoint
from model_intent import load_model_checkpoint as load_intent_checkpoint

# Configuration
CACHE_DIR = pathlib.Path("/Users/fusg/Documents/gfp-fork/dataset_cache/")
CKPT_PATH = "../models/cfm_base.pt"
USE_INTENT_MODEL = None  # None=auto, True=force intent model, False=force base model
PARQUET_PATH = CACHE_DIR / "trajs_LSAS_filtered.parquet"
CACHE_KEY_FILE = "87991419c1be4279.key.json"
OUTPUT_STRIDE_SECONDS = 5

# Device configuration
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

Using device: mps


## Data Loading

In [2]:
# Load cached dataset artifacts
def load_cache_key():
    """Load dataset cache key file."""
    if CACHE_KEY_FILE:
        key_path = Path(CACHE_KEY_FILE)
        if not key_path.is_absolute():
            key_path = CACHE_DIR / key_path
        if not key_path.exists():
            raise FileNotFoundError(
                f"Specified cache key file not found: {key_path}. "
                f"Make sure it exists under {CACHE_DIR}."
            )
    else:
        key_files = sorted(
            CACHE_DIR.glob("*.key.json"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        if not key_files:
            raise FileNotFoundError(
                "No dataset_cache key files found. Run the training notebook first."
            )
        key_path = key_files[0]
    return key_path

# Load cache key and dataset information
key_path = load_cache_key()
key_info = json.loads(key_path.read_text())
print(f"[cache] using key: {key_path.name}")

dset_key = key_info["dataset_key"]
stats_key = key_info["stats_key"]
paths = cache_paths(dset_key, stats_key)

# Load dataset arrays (using memory-mapped files for efficiency)
X_train = np.load(paths["x_tr"], mmap_mode="r")
Y_train = np.load(paths["y_tr"], mmap_mode="r")
C_train = np.load(paths["c_tr"], mmap_mode="r")
X_val = np.load(paths["x_va"], mmap_mode="r")
Y_val = np.load(paths["y_va"], mmap_mode="r")
C_val = np.load(paths["c_va"], mmap_mode="r")
X_test = np.load(paths["x_te"], mmap_mode="r")
Y_test = np.load(paths["y_te"], mmap_mode="r")
C_test = np.load(paths["c_te"], mmap_mode="r")

# Load normalization statistics and metadata
norm_stats = json.loads(paths["stats"].read_text())
meta_train = pd.read_parquet(paths["meta_tr"])  # per-window metadata
meta_val = pd.read_parquet(paths["meta_va"])
meta_test = pd.read_parquet(paths["meta_te"])
manifest = json.loads(paths["manifest"].read_text())
summary = json.loads(paths["summary"].read_text())

# Extract normalization parameters
feat_mean = norm_stats["feat_mean"]
feat_std = norm_stats["feat_std"]
ctx_mean = norm_stats["ctx_mean"]
ctx_std = norm_stats["ctx_std"]

# Display dataset sizes
dataset_sizes = {
    k: (int(v) if isinstance(v, (int, np.integer)) else v)
    for k, v in summary.get("sizes", {}).items()
}
print("Dataset sizes:", dataset_sizes)

# Load trajectory data
trajs = Traffic.from_file(PARQUET_PATH)
print(f"Loaded {trajs.data.flight_id.nunique()} trajectories from {PARQUET_PATH}")

[cache] using key: 87991419c1be4279.key.json
Dataset sizes: {'train': 2000000, 'val': 500000, 'test': 400000}
Loaded 178947 trajectories from /Users/fusg/Documents/gfp-fork/dataset_cache/trajs_LSAS_filtered.parquet


## Model Architecture

Import the Flow Matching Model components for trajectory prediction.

In [3]:
# Import model architecture from dedicated module
from model import load_model_checkpoint as load_base_checkpoint
from model_intent import load_model_checkpoint as load_intent_checkpoint, compute_intent_features_batch

# Helper to select the correct checkpoint loader and prepare intent context.
def load_model_checkpoint(checkpoint_path, device, use_intent_model=None):
    ckpt = torch.load(checkpoint_path, map_location=device)
    cfg = ckpt.get("model_cfg", {})
    if use_intent_model is None:
        use_intent_model = int(cfg.get("context_dim", 8)) != 8
    if use_intent_model:
        model = load_intent_checkpoint(checkpoint_path, device)
    else:
        model = load_base_checkpoint(checkpoint_path, device)
    print("Using IntentFlowMatchingModel" if use_intent_model else "Using FlowMatchingModel")
    return model, ckpt.get("preprocess", {})


def _ctx_from_intent(x_hist, ctx, preprocess):
    if not preprocess or not preprocess.get("intent_keep_idx"):
        return ctx
    intent_keep_idx = preprocess["intent_keep_idx"]
    intent_mean = np.asarray(preprocess["intent_feat_mean"], dtype=np.float32)
    intent_std = np.asarray(preprocess["intent_feat_std"], dtype=np.float32)
    feat_mean_arr = np.asarray(preprocess.get("feat_mean", feat_mean), dtype=np.float32)
    feat_std_arr = np.asarray(preprocess.get("feat_std", feat_std), dtype=np.float32)
    x_np = x_hist.detach().cpu().numpy() if torch.is_tensor(x_hist) else np.asarray(x_hist)
    if x_np.ndim == 2:
        x_np = x_np[None, ...]
    x_phys = x_np * feat_std_arr[None, None, :] + feat_mean_arr[None, None, :]
    intent_full = compute_intent_features_batch(x_phys)
    intent_sel = intent_full[:, intent_keep_idx]
    intent_norm = (intent_sel - intent_mean[None, :]) / intent_std[None, :]
    intent_t = torch.from_numpy(intent_norm).to(ctx.device).type_as(ctx)
    return torch.cat([ctx, intent_t], dim=1)


def _ctx_base(ctx):
    return ctx[:, :8] if ctx.size(-1) > 8 else ctx

## Model Loading

In [4]:
# Load the pre-trained model
model, CKPT_PREPROCESS = load_model_checkpoint(CKPT_PATH, DEVICE, USE_INTENT_MODEL)
print(f"Loaded checkpoint: {CKPT_PATH}")

# If the checkpoint contains its own preprocessing stats, renormalize
# the cached dataset arrays to match the checkpoint's feature/context scaling.
if CKPT_PREPROCESS and CKPT_PREPROCESS.get("feat_mean") is not None:
    feat_mean_ckpt = np.asarray(CKPT_PREPROCESS["feat_mean"], dtype=np.float32)
    feat_std_ckpt = np.asarray(CKPT_PREPROCESS["feat_std"], dtype=np.float32)
    ctx_mean_ckpt = np.asarray(CKPT_PREPROCESS["ctx_mean"], dtype=np.float32)
    ctx_std_ckpt = np.asarray(CKPT_PREPROCESS["ctx_std"], dtype=np.float32)

    feat_mean_arr = np.asarray(feat_mean, dtype=np.float32)
    feat_std_arr = np.asarray(feat_std, dtype=np.float32)
    ctx_mean_arr = np.asarray(ctx_mean, dtype=np.float32)
    ctx_std_arr = np.asarray(ctx_std, dtype=np.float32)

    X_test = (X_test * feat_std_arr[None, None, :]) + feat_mean_arr[None, None, :]
    Y_test = (Y_test * feat_std_arr[None, None, :]) + feat_mean_arr[None, None, :]
    C_test = (C_test * ctx_std_arr[None, :]) + ctx_mean_arr[None, :]

    X_test = (X_test - feat_mean_ckpt[None, None, :]) / feat_std_ckpt[None, None, :]
    Y_test = (Y_test - feat_mean_ckpt[None, None, :]) / feat_std_ckpt[None, None, :]
    C_test = (C_test - ctx_mean_ckpt[None, :]) / ctx_std_ckpt[None, :]

    feat_mean = feat_mean_ckpt
    feat_std = feat_std_ckpt
    ctx_mean = ctx_mean_ckpt
    ctx_std = ctx_std_ckpt
    print("Renormalized dataset arrays using checkpoint preprocessing stats.")

Using FlowMatchingModel
Loaded checkpoint: ../models/cfm_base.pt


/Users/fusg/Documents/gfp-fork/model.py:77: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc_layer, num_layers)


## Sample Generation and Visualization

Generate trajectory predictions and create visualizations.

In [43]:
# Choose trajectories for analysis
CASE_LIST = [169947] #np.random.choice(X_test.shape[0], size=3, replace=False).tolist() # [6841, 10000, 12345]   # specific trajectory indices
N_TRAJS = len(CASE_LIST)
N_SAMPLES = 64
TIMESTEPS = int(Y_test.shape[1])

# Prepare input data
x_hist_batch = torch.from_numpy(np.array([X_test[idx] for idx in CASE_LIST])).to(DEVICE).contiguous()
ctx_batch = torch.from_numpy(np.array([C_test[idx] for idx in CASE_LIST])).to(DEVICE).contiguous()
ctx_batch_orig = ctx_batch
ctx_batch = _ctx_from_intent(x_hist_batch, ctx_batch, CKPT_PREPROCESS)

print(f"Processing {N_TRAJS} trajectories with {N_SAMPLES} samples each")

# Generate future trajectory samples
y_norm_all = sample_many(
    model,
    x_hist_batch,
    ctx_batch,
    T_out=TIMESTEPS,
    n_steps=64,
    G=1.0,
    n_samples=N_SAMPLES,
    chunk=128,
)

# Convert back to global coordinates
y_glob_all = (
    denorm_seq_to_global(
        y_norm_all,
        ctx_batch_orig.repeat(N_SAMPLES, 1),
        feat_mean, feat_std, ctx_mean, ctx_std,
    )
    .cpu()
    .numpy()
    .reshape(N_SAMPLES, N_TRAJS, TIMESTEPS, -1)
)

# Get history and ground truth in global coordinates
x_hist_glob = (
    denorm_seq_to_global(
        x_hist_batch, ctx_batch_orig, feat_mean, feat_std, ctx_mean, ctx_std
    )
    .cpu()
    .numpy()[:, :, :3]
)

y_true_glob = (
    denorm_seq_to_global(
        torch.from_numpy(np.array([Y_test[idx] for idx in CASE_LIST])).to(DEVICE),
        ctx_batch_orig, feat_mean, feat_std, ctx_mean, ctx_std,
    )
    .cpu()
    .numpy()[:, :, :3]
)

print(f"Generated predictions with shape: {y_glob_all.shape}")
print(f"History shape: {x_hist_glob.shape}, Ground truth shape: {y_true_glob.shape}")

Processing 1 trajectories with 64 samples each
Generated predictions with shape: (64, 1, 12, 7)
History shape: (1, 60, 3), Ground truth shape: (1, 12, 3)


In [39]:
CASE_LIST

[121270, 169947, 117978]

## Coordinate System Conversion

Convert between different coordinate systems for visualization.

In [41]:
# Coordinate transformation setup
crs_lv95 = pyproj.CRS.from_epsg(2056)  # Swiss LV95
crs_wgs84 = pyproj.CRS.from_epsg(4326)  # WGS84 (lat/lon)
to_wgs84 = pyproj.Transformer.from_crs(crs_lv95, crs_wgs84, always_xy=True)

def xy_to_lonlat_arr(xy: np.ndarray) -> np.ndarray:
    """Convert XY coordinates to longitude/latitude."""
    lons, lats = to_wgs84.transform(xy[:, 0], xy[:, 1])
    return np.stack([lons, lats], axis=1)

def concat_polyline_lonlat(list_of_ll_arrays):
    """Concatenate multiple lon/lat arrays with None separators for Plotly."""
    lons, lats = [], []
    for arr in list_of_ll_arrays:
        lons.extend(arr[:, 0].tolist())
        lats.extend(arr[:, 1].tolist())
        lons.append(None)
        lats.append(None)
    if lons: lons.pop()  # Remove last None
    if lats: lats.pop()  # Remove last None
    return lons, lats

## Spaghetti Plot Visualization

Create spaghetti plots showing multiple trajectory predictions.

In [44]:
# Color scheme for plots
COLOR_HISTORY = "black"
COLOR_GT = "red"
COLOR_PRED = "#1f77b4"
COLOR_MEAN = "#e19f20"

# Create subplot grid
cols = min(3, N_TRAJS)
fig_spaghetti = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "map"} for _ in range(3)]],
    subplot_titles=[
        f"{meta_test.loc[CASE_LIST[i]]['flight_id'].split('_')[1]}" if i < N_TRAJS else "—"
        for i in range(3)
    ],
    horizontal_spacing=0.02,
)

for b in range(cols):
    # Convert coordinates to lon/lat
    hist_xy = x_hist_glob[b, :, :2]
    true_xy = y_true_glob[b, :, :2]
    samples_xy = [y_glob_all[s, b, :, :2] for s in range(N_SAMPLES)]

    hist_ll = xy_to_lonlat_arr(hist_xy)
    true_ll = xy_to_lonlat_arr(true_xy)
    samples_ll = [xy_to_lonlat_arr(sxy) for sxy in samples_xy]

    # Center map on trajectory
    all_ll = np.vstack([hist_ll, true_ll])
    lon_center = float((np.nanmin(all_ll[:, 0]) + np.nanmax(all_ll[:, 0])) * 0.5)
    lat_center = float((np.nanmin(all_ll[:, 1]) + np.nanmax(all_ll[:, 1])) * 0.5)

    # Concatenate samples into one polyline trace
    samp_lon, samp_lat = concat_polyline_lonlat(samples_ll)

    showleg = (b == 0)

    # Predicted samples (blue, semi-transparent spaghetti)
    fig_spaghetti.add_trace(
        go.Scattermap(
            lon=samp_lon, lat=samp_lat,
            mode="lines",
            line=dict(width=1, color=COLOR_PRED),
            opacity=0.5,
            name=f"{N_SAMPLES} samples", showlegend=showleg
        ),
        row=1, col=b+1
    )

    # Sample mean (orange)
    mean_ll = np.mean(np.stack(samples_ll, axis=0), axis=0)
    fig_spaghetti.add_trace(
        go.Scattermap(
            lon=mean_ll[:, 0], lat=mean_ll[:, 1],
            mode="lines",
            line=dict(width=3, color=COLOR_MEAN),
            name="Sample mean", showlegend=showleg
        ),
        row=1, col=b+1
    )

    # History (black)
    fig_spaghetti.add_trace(
        go.Scattermap(
            lon=hist_ll[:, 0], lat=hist_ll[:, 1],
            mode="lines",
            line=dict(width=1.5, color=COLOR_HISTORY),
            name="History", showlegend=showleg
        ),
        row=1, col=b+1
    )

    # Ground Truth (red, on top)
    fig_spaghetti.add_trace(
        go.Scattermap(
            lon=true_ll[:, 0], lat=true_ll[:, 1],
            mode="lines+markers",
            line=dict(width=1, color=COLOR_GT),
            marker=dict(size=6, color=COLOR_GT),
            name="Ground Truth", showlegend=showleg
        ),
        row=1, col=b+1
    )

    # Map settings
    key = "" if b == 0 else str(b+1)
    fig_spaghetti.update_layout({
        f"map{key}": dict(
            style="carto-positron",
            center=dict(lon=lon_center, lat=lat_center),
            zoom=10.5,
        )
    })

# Fill empty panels if fewer than 3
for k in range(2, 4):
    if k > cols:
        fig_spaghetti.update_layout({f"map{k}": dict(style="carto-positron")})

fig_spaghetti.update_layout(
    margin=dict(l=10, r=10, t=50, b=10),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.15,
        xanchor="center",
        x=0.5
    )
)

# Save and display
#fig_spaghetti.write_html('../figures/spaghetti.html')
fig_spaghetti.show()

In [38]:
# Color scheme for plots
COLOR_HISTORY = "black"
COLOR_GT = "red"
COLOR_PRED = "#1f77b4"
COLOR_MEAN = "#e19f20"

# Create subplot grid
cols = min(3, N_TRAJS)
fig_spaghetti = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "map"} for _ in range(3)]],
    subplot_titles=[
        f"{meta_test.loc[CASE_LIST[i]]['flight_id'].split('_')[1]}" if i < N_TRAJS else "—"
        for i in range(3)
    ],
    horizontal_spacing=0.02,
)

for b in range(cols):
    # Convert coordinates to lon/lat
    hist_xy = x_hist_glob[b, :, :2]
    true_xy = y_true_glob[b, :, :2]
    samples_xy = [y_glob_all[s, b, :, :2] for s in range(N_SAMPLES)]

    hist_ll = xy_to_lonlat_arr(hist_xy)
    true_ll = xy_to_lonlat_arr(true_xy)
    samples_ll = [xy_to_lonlat_arr(sxy) for sxy in samples_xy]

    # Center map on trajectory
    all_ll = np.vstack([hist_ll, true_ll])
    lon_center = float((np.nanmin(all_ll[:, 0]) + np.nanmax(all_ll[:, 0])) * 0.5)
    lat_center = float((np.nanmin(all_ll[:, 1]) + np.nanmax(all_ll[:, 1])) * 0.5)

    # Concatenate samples into one polyline trace
    samp_lon, samp_lat = concat_polyline_lonlat(samples_ll)

    showleg = (b == 0)

    # Predicted samples (blue, semi-transparent spaghetti)
    fig_spaghetti.add_trace(
        go.Scattermap(
            lon=samp_lon, lat=samp_lat,
            mode="lines",
            line=dict(width=1, color=COLOR_PRED),
            opacity=0.5,
            name=f"{N_SAMPLES} samples", showlegend=showleg
        ),
        row=1, col=b+1
    )

    # Sample mean (orange)
    mean_ll = np.mean(np.stack(samples_ll, axis=0), axis=0)
    fig_spaghetti.add_trace(
        go.Scattermap(
            lon=mean_ll[:, 0], lat=mean_ll[:, 1],
            mode="lines",
            line=dict(width=3, color=COLOR_MEAN),
            name="Sample mean", showlegend=showleg
        ),
        row=1, col=b+1
    )

    # History (black)
    fig_spaghetti.add_trace(
        go.Scattermap(
            lon=hist_ll[:, 0], lat=hist_ll[:, 1],
            mode="lines",
            line=dict(width=1.5, color=COLOR_HISTORY),
            name="History", showlegend=showleg
        ),
        row=1, col=b+1
    )

    # Ground Truth (red, on top)
    fig_spaghetti.add_trace(
        go.Scattermap(
            lon=true_ll[:, 0], lat=true_ll[:, 1],
            mode="lines+markers",
            line=dict(width=1, color=COLOR_GT),
            marker=dict(size=6, color=COLOR_GT),
            name="Ground Truth", showlegend=showleg
        ),
        row=1, col=b+1
    )

    # Map settings
    key = "" if b == 0 else str(b+1)
    fig_spaghetti.update_layout({
        f"map{key}": dict(
            style="carto-positron",
            center=dict(lon=lon_center, lat=lat_center),
            zoom=10.5,
        )
    })

# Fill empty panels if fewer than 3
for k in range(2, 4):
    if k > cols:
        fig_spaghetti.update_layout({f"map{k}": dict(style="carto-positron")})

fig_spaghetti.update_layout(
    margin=dict(l=10, r=10, t=50, b=10),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.15,
        xanchor="center",
        x=0.5
    )
)

# Save and display
#fig_spaghetti.write_html('../figures/spaghetti.html')
fig_spaghetti.show()

## Physics Consistency Check

Measure how closely generated trajectories satisfy \(\Delta p \approx v \Delta t\) on a small test slice. This is a lightweight diagnostic to decide whether retraining with the physics term is worthwhile.

In [48]:
import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Statistical consistency check on a larger random slice, kept bounded for speed.
RNG = np.random.default_rng(42)
CHECK_CASE_COUNT = min(48, len(X_test))
CHECK_SAMPLES = 16
CHECK_STEPS = 12
CHECK_DT_SECONDS = OUTPUT_STRIDE_SECONDS
CHECK_CASES = RNG.choice(len(X_test), size=CHECK_CASE_COUNT, replace=False).tolist()

x_check = torch.from_numpy(np.asarray(X_test[CHECK_CASES])).to(DEVICE).contiguous()
ctx_check = torch.from_numpy(np.asarray(C_test[CHECK_CASES])).to(DEVICE).contiguous()
y_check = torch.from_numpy(np.asarray(Y_test[CHECK_CASES])).to(DEVICE).contiguous()

with torch.no_grad():
    y_check_norm = sample_many(
        model,
        x_check,
        ctx_check,
        T_out=int(Y_test.shape[1]),
        n_steps=CHECK_STEPS,
        G=1.0,
        n_samples=CHECK_SAMPLES,
        chunk=4,
    )
    y_check_glob = denorm_seq_to_global(
        y_check_norm,
        ctx_check.repeat(CHECK_SAMPLES, 1),
        feat_mean,
        feat_std,
        ctx_mean,
        ctx_std,
    ).cpu().numpy().reshape(CHECK_SAMPLES, CHECK_CASE_COUNT, int(Y_test.shape[1]), -1)

# Generated trajectories: physics residual over horizon
pred_pos = y_check_glob[..., :3]
pred_vel = y_check_glob[..., 3:6]
pred_dpos = pred_pos[:, :, 1:, :] - pred_pos[:, :, :-1, :]
pred_expected = pred_vel[:, :, :-1, :] * float(CHECK_DT_SECONDS)
pred_residual = pred_dpos - pred_expected
pred_residual_norm = np.linalg.norm(pred_residual, axis=-1)
pred_relative_residual = pred_residual_norm / (np.linalg.norm(pred_dpos, axis=-1) + 1e-6)

# Ground-truth trajectories on the same cases
y_true_glob = denorm_seq_to_global(
    y_check,
    ctx_check,
    feat_mean,
    feat_std,
    ctx_mean,
    ctx_std,
).cpu().numpy()

gt_pos = y_true_glob[..., :3]
gt_vel = y_true_glob[..., 3:6]
gt_dpos = gt_pos[:, 1:, :] - gt_pos[:, :-1, :]
gt_expected = gt_vel[:, :-1, :] * float(CHECK_DT_SECONDS)
gt_residual = gt_dpos - gt_expected
gt_residual_norm = np.linalg.norm(gt_residual, axis=-1)
gt_relative_residual = gt_residual_norm / (np.linalg.norm(gt_dpos, axis=-1) + 1e-6)

# Horizon-wise summary statistics
horizons_s = (np.arange(1, int(Y_test.shape[1])) * float(CHECK_DT_SECONDS)).astype(int)

def summarize_pred(curve_3d: np.ndarray):
    flat = curve_3d.reshape(-1, curve_3d.shape[-1])
    return {
        "mean": flat.mean(axis=0),
        "median": np.median(flat, axis=0),
        "p10": np.percentile(flat, 10, axis=0),
        "p90": np.percentile(flat, 90, axis=0),
    }

def summarize_gt(curve_2d: np.ndarray):
    return {
        "mean": curve_2d.mean(axis=0),
        "median": np.median(curve_2d, axis=0),
        "p10": np.percentile(curve_2d, 10, axis=0),
        "p90": np.percentile(curve_2d, 90, axis=0),
    }

pred_sum = summarize_pred(pred_residual_norm)
gt_sum = summarize_gt(gt_residual_norm)
pred_rel_sum = summarize_pred(pred_relative_residual)
gt_rel_sum = summarize_gt(gt_relative_residual)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=(
        f"Position-vs-velocity residual over horizon ({CHECK_CASE_COUNT} test cases × {CHECK_SAMPLES} samples)",
        "Relative residual over horizon",
    ),
)

# Row 1: absolute residual norm
fig.add_trace(
    go.Scatter(
        x=horizons_s,
        y=pred_sum["mean"],
        mode="lines+markers",
        name="Generated mean residual",
        line=dict(color="#1f77b4", width=2),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=np.concatenate([horizons_s, horizons_s[::-1]]),
        y=np.concatenate([pred_sum["p10"], pred_sum["p90"][::-1]]),
        fill="toself",
        fillcolor="rgba(31, 119, 180, 0.15)",
        line=dict(color="rgba(31, 119, 180, 0)"),
        hoverinfo="skip",
        showlegend=False,
        name="Generated 10-90%",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=horizons_s,
        y=gt_sum["mean"],
        mode="lines+markers",
        name="Ground truth mean residual",
        line=dict(color="#d62728", width=2),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=np.concatenate([horizons_s, horizons_s[::-1]]),
        y=np.concatenate([gt_sum["p10"], gt_sum["p90"][::-1]]),
        fill="toself",
        fillcolor="rgba(214, 39, 40, 0.12)",
        line=dict(color="rgba(214, 39, 40, 0)"),
        hoverinfo="skip",
        showlegend=False,
        name="Ground truth 10-90%",
    ),
    row=1,
    col=1,
)

# Row 2: relative residual
fig.add_trace(
    go.Scatter(
        x=horizons_s,
        y=pred_rel_sum["mean"],
        mode="lines+markers",
        name="Generated mean relative residual",
        line=dict(color="#1f77b4", width=2),
        showlegend=False,
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=np.concatenate([horizons_s, horizons_s[::-1]]),
        y=np.concatenate([pred_rel_sum["p10"], pred_rel_sum["p90"][::-1]]),
        fill="toself",
        fillcolor="rgba(31, 119, 180, 0.15)",
        line=dict(color="rgba(31, 119, 180, 0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=horizons_s,
        y=gt_rel_sum["mean"],
        mode="lines+markers",
        name="Ground truth mean relative residual",
        line=dict(color="#d62728", width=2),
        showlegend=False,
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=np.concatenate([horizons_s, horizons_s[::-1]]),
        y=np.concatenate([gt_rel_sum["p10"], gt_rel_sum["p90"][::-1]]),
        fill="toself",
        fillcolor="rgba(214, 39, 40, 0.12)",
        line=dict(color="rgba(214, 39, 40, 0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig.update_layout(
    template="plotly_white",
    height=750,
    width=1050,
    title="Physics consistency check for cfm_base.pt",
    legend=dict(orientation="h", yanchor="bottom", y=-0.12, xanchor="center", x=0.5),
    margin=dict(l=20, r=20, t=80, b=40),
)
fig.update_xaxes(title_text="Horizon [s]", row=2, col=1)
fig.update_yaxes(title_text="Residual [m]", row=1, col=1)
fig.update_yaxes(title_text="Relative residual", row=2, col=1)

print(
    {
        "device": str(DEVICE),
        "cases": CHECK_CASE_COUNT,
        "samples_per_case": CHECK_SAMPLES,
        "n_steps": CHECK_STEPS,
        "predicted_mean_residual_m": float(np.mean(pred_residual_norm)),
        "predicted_median_residual_m": float(np.median(pred_residual_norm)),
        "predicted_mean_relative_residual": float(np.mean(pred_relative_residual)),
        "ground_truth_mean_residual_m": float(np.mean(gt_residual_norm)),
        "ground_truth_median_residual_m": float(np.median(gt_residual_norm)),
        "ground_truth_mean_relative_residual": float(np.mean(gt_relative_residual)),
    }
)

fig.show()

{'device': 'mps', 'cases': 48, 'samples_per_case': 16, 'n_steps': 12, 'predicted_mean_residual_m': 339.5366516113281, 'predicted_median_residual_m': 277.50604248046875, 'predicted_mean_relative_residual': 0.5757129192352295, 'ground_truth_mean_residual_m': 14.859186172485352, 'ground_truth_median_residual_m': 12.628068923950195, 'ground_truth_mean_relative_residual': 0.012738735415041447}


In [50]:
from tqdm.auto import tqdm

@torch.no_grad()
def constant_velocity_extrapolation(
    X_test,
    C_test,
    feat_mean,
    feat_std,
    ctx_mean,
    ctx_std,
    device,
    T_out: int,
    dt_seconds: float = OUTPUT_STRIDE_SECONDS,
):
    """Deterministic constant-velocity baseline in global coordinates."""
    # Denorm history to global to read last state
    x_hist = torch.from_numpy(np.array(X_test)).to(device).contiguous()
    ctx = torch.from_numpy(np.array(C_test)).to(device).contiguous()
    x_hist_glob = (
        denorm_seq_to_global(x_hist, ctx, feat_mean, feat_std, ctx_mean, ctx_std)
        .cpu()
        .numpy()
    )  # (N, L, D)

    N, L, D = x_hist_glob.shape
    pos0 = x_hist_glob[:, -1, :3]  # (N, 3): x,y,z at last history step
    vel0 = np.zeros((N, 3), dtype=pos0.dtype)
    if D >= 6:
        vel0 = x_hist_glob[:, -1, 3:6]  # (N, 3): vx,vy,vz in m/s

    # Prepare output (N, T_out, D)
    y_pred_glob = np.zeros((N, T_out, D), dtype=x_hist_glob.dtype)

    # Fill positions by extrapolation, velocities constant; other channels set to 0
    tgrid = np.arange(1, T_out + 1, dtype=pos0.dtype) * dt_seconds  # seconds ahead
    disp = tgrid[None, :, None] * vel0[:, None, :]  # (N,T,3)

    # Positions
    y_pred_glob[:, :, 0] = pos0[:, 0][:, None] + disp[:, :, 0]  # x
    y_pred_glob[:, :, 1] = pos0[:, 1][:, None] + disp[:, :, 1]  # y
    if D >= 3:
        y_pred_glob[:, :, 2] = pos0[:, 2][:, None] + disp[:, :, 2]  # z

    # Velocities
    if D >= 4:
        y_pred_glob[:, :, 3] = vel0[:, 0][:, None]  # vx
    if D >= 5:
        y_pred_glob[:, :, 4] = vel0[:, 1][:, None]  # vy
    if D >= 6:
        y_pred_glob[:, :, 5] = vel0[:, 2][:, None]  # vz

    # Any remaining channels set to 0
    if D > 6:
        y_pred_glob[:, :, 6:] = 0.0

    return y_pred_glob  # global coords

@torch.no_grad()
def rmse_mae_3d_2d_vert_vs_horizon(
    model,
    X,
    Y,
    C,
    feat_mean,
    feat_std,
    ctx_mean,
    ctx_std,
    device,
    preprocess=None,
    n_eval: int = 2000,
    n_samples: int = 32,
    n_steps: int = 64,
    batch_size: int = 128,
    dt_seconds: float = OUTPUT_STRIDE_SECONDS,
    progress: bool = True,
    desc: str = "Evaluating batches",
):
    """Single pass evaluator for RMSE & MAE vs horizon."""
    N = min(int(X.shape[0]), int(n_eval))
    T = int(Y.shape[1])

    def zeros():
        return np.zeros(T, dtype=np.float64)

    # Accumulators
    acc = {
        "rmse": {
            "3d": {"mean": zeros(), "best": zeros(), "cv": zeros()},
            "2d": {"mean": zeros(), "best": zeros(), "cv": zeros()},
            "vert": {"mean": zeros(), "best": zeros(), "cv": zeros()},
        },
        "mae": {
            "3d": {"mean": zeros(), "best": zeros(), "cv": zeros()},
            "2d": {"mean": zeros(), "best": zeros(), "cv": zeros()},
            "vert": {"mean": zeros(), "best": zeros(), "cv": zeros()},
        },
    }
    num_cases = 0

    # Progress tracking
    num_batches = (N + batch_size - 1) // batch_size
    iterator = range(0, N, batch_size)
    if progress:
        iterator = tqdm(iterator, total=num_batches, desc=desc, leave=True)

    for i0 in iterator:
        i1 = min(N, i0 + batch_size)

        # Tensors on device
        x_hist = torch.from_numpy(np.array(X[i0:i1])).to(device).contiguous()
        ctx = torch.from_numpy(np.array(C[i0:i1])).to(device).contiguous()
        y_true = torch.from_numpy(np.array(Y[i0:i1])).to(device).contiguous()
        B = int(x_hist.shape[0])
        ctx_model = _ctx_from_intent(x_hist, ctx, preprocess)

        # Samples (normalized) → global
        y_norm_all = sample_many(
            model, x_hist, ctx_model,
            T_out=T, n_steps=n_steps, G=1.0,
            n_samples=n_samples, chunk=128,
        )  # (S*B, T, D_norm)

        y_s_glob = denorm_seq_to_global(
            y_norm_all, ctx.repeat(n_samples, 1),
            feat_mean, feat_std, ctx_mean, ctx_std
        ).view(n_samples, B, T, -1)  # (S,B,T,D)

        # Ground truth global
        y_true_glob = denorm_seq_to_global(
            y_true, ctx, feat_mean, feat_std, ctx_mean, ctx_std
        )  # (B,T,D)

        # Mean prediction across samples
        y_mean_glob = y_s_glob.mean(dim=0)  # (B,T,D)

        # Errors for mean
        dx_mean = y_mean_glob[..., 0] - y_true_glob[..., 0]  # (B,T)
        dy_mean = y_mean_glob[..., 1] - y_true_glob[..., 1]
        dz_mean = y_mean_glob[..., 2] - y_true_glob[..., 2]

        dist3_mean_sq = dx_mean**2 + dy_mean**2 + dz_mean**2
        dist2_mean_sq = dx_mean**2 + dy_mean**2
        vert_mean_sq = dz_mean**2

        dist3_mean = torch.sqrt(dist3_mean_sq + 1e-12)
        dist2_mean = torch.sqrt(dist2_mean_sq + 1e-12)
        vert_mean = torch.sqrt(vert_mean_sq + 1e-12)  # = |dz|

        # Errors for best-of-N
        dx_all = y_s_glob[..., 0] - y_true_glob.unsqueeze(0)[..., 0]  # (S,B,T)
        dy_all = y_s_glob[..., 1] - y_true_glob.unsqueeze(0)[..., 1]
        dz_all = y_s_glob[..., 2] - y_true_glob.unsqueeze(0)[..., 2]

        dist3_all_sq = dx_all**2 + dy_all**2 + dz_all**2       # (S,B,T)
        dist2_all_sq = dx_all**2 + dy_all**2                   # (S,B,T)
        vert_all_sq = dz_all**2                               # (S,B,T)

        # Min over samples (S) per (B,T)
        dist3_best_sq = dist3_all_sq.min(dim=0).values
        dist2_best_sq = dist2_all_sq.min(dim=0).values
        vert_best_sq = vert_all_sq.min(dim=0).values

        dist3_best = torch.sqrt(dist3_best_sq + 1e-12)
        dist2_best = torch.sqrt(dist2_best_sq + 1e-12)
        vert_best = torch.sqrt(vert_best_sq + 1e-12)  # = min |dz|

        # Constant-velocity baseline
        cv_pred_glob = constant_velocity_extrapolation(
            X[i0:i1], C[i0:i1],
            feat_mean, feat_std, ctx_mean, ctx_std,
            device=device, T_out=T, dt_seconds=dt_seconds,
        )  # (B,T,D)
        cv = torch.from_numpy(cv_pred_glob).to(device)

        dx_cv = cv[..., 0] - y_true_glob[..., 0]
        dy_cv = cv[..., 1] - y_true_glob[..., 1]
        dz_cv = cv[..., 2] - y_true_glob[..., 2]

        dist3_cv_sq = dx_cv**2 + dy_cv**2 + dz_cv**2
        dist2_cv_sq = dx_cv**2 + dy_cv**2
        vert_cv_sq = dz_cv**2

        dist3_cv = torch.sqrt(dist3_cv_sq + 1e-12)
        dist2_cv = torch.sqrt(dist2_cv_sq + 1e-12)
        vert_cv = torch.sqrt(vert_cv_sq + 1e-12)  # = |dz|

        # Accumulate (sum over batch; average over num_cases later)
        for key, sq_t, abs_t in [
            (("3d","mean"), dist3_mean_sq, dist3_mean),
            (("2d","mean"), dist2_mean_sq, dist2_mean),
            (("vert","mean"), vert_mean_sq, vert_mean),
            (("3d","best"), dist3_best_sq, dist3_best),
            (("2d","best"), dist2_best_sq, dist2_best),
            (("vert","best"), vert_best_sq, vert_best),
            (("3d","cv"), dist3_cv_sq, dist3_cv),
            (("2d","cv"), dist2_cv_sq, dist2_cv),
            (("vert","cv"), vert_cv_sq, vert_cv),
        ]:
            cat, est = key
            acc["rmse"][cat][est] += sq_t.detach().cpu().numpy().sum(axis=0)
            acc["mae"][cat][est]  += abs_t.detach().cpu().numpy().sum(axis=0)

        num_cases += B

        # Progress update
        if progress:
            iterator.set_postfix_str(f"cases={num_cases}, B={B}")

    # Finalize
    denom = max(1, num_cases)
    out = {"horizons_s": (np.arange(1, T + 1) * dt_seconds).astype(int), "rmse": {}, "mae": {}, "fde": {}}
    for cat in ["3d", "2d", "vert"]:
        out["rmse"][cat] = {}
        out["mae"][cat]  = {}
        out["fde"][cat]  = {}
        for est in ["mean", "best", "cv"]:
            rmse_curve = np.sqrt(acc["rmse"][cat][est] / denom)
            mae_curve  = acc["mae"][cat][est] / denom
            out["rmse"][cat][est] = rmse_curve
            out["mae"][cat][est]  = mae_curve
            out["fde"][cat][est] = {
                "rmse": rmse_curve[-1],
                "mae":  mae_curve[-1],
            }
    return out

# Run evaluation metrics
res = rmse_mae_3d_2d_vert_vs_horizon(
    model,
    X_test, Y_test, C_test,
    feat_mean, feat_std, ctx_mean, ctx_std,
    device=DEVICE,
    preprocess=CKPT_PREPROCESS,
    n_eval=128,  
    n_samples=64,
    n_steps=32,
    batch_size=16,
    dt_seconds=OUTPUT_STRIDE_SECONDS
)

print("Evaluation completed successfully")

Evaluating batches:   0%|          | 0/8 [00:00<?, ?it/s]

Evaluation completed successfully


In [51]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

H = res["horizons_s"]

# --- build 2-column × 3-row subplot ---
fig = make_subplots(
    rows=3, cols=2,
    shared_xaxes=True,
    subplot_titles=[
        "MAE — 3D (x,y,z)", "RMSE — 3D (x,y,z)",
        "MAE — 2D horizontal (x,y)", "RMSE — 2D horizontal (x,y)",
        "MAE — Vertical (z)", "RMSE — Vertical (z)"
    ],
    vertical_spacing=0.08,
    horizontal_spacing=0.08
)

# === color scheme ===
COLOR_MEAN = "blue"
COLOR_BEST = "green"
COLOR_CV   = "red"

def add_metric_traces(cat_name, row_mae, row_rmse, show_legend=False):
    # --- MAE (left) ---
    fig.add_trace(go.Scatter(
        x=H, y=res["mae"][cat_name]["mean"],
        name="Model (mean)", mode="lines+markers",
        line=dict(width=2, color=COLOR_MEAN),
        showlegend=show_legend
    ), row=row_mae, col=1)

    fig.add_trace(go.Scatter(
        x=H, y=res["mae"][cat_name]["best"],
        name="Model (best-of-S)", mode="lines+markers",
        line=dict(width=2, dash="dash", color=COLOR_BEST),
        showlegend=show_legend
    ), row=row_mae, col=1)

    fig.add_trace(go.Scatter(
        x=H, y=res["mae"][cat_name]["cv"],
        name="Constant velocity", mode="lines+markers",
        line=dict(width=2, dash="dot", color=COLOR_CV),
        showlegend=show_legend
    ), row=row_mae, col=1)

    # --- RMSE (right) ---
    fig.add_trace(go.Scatter(
        x=H, y=res["rmse"][cat_name]["mean"],
        name="Model (mean)", mode="lines+markers",
        line=dict(width=2, color=COLOR_MEAN),
        showlegend=False
    ), row=row_rmse, col=2)

    fig.add_trace(go.Scatter(
        x=H, y=res["rmse"][cat_name]["best"],
        name="Model (best-of-S)", mode="lines+markers",
        line=dict(width=2, dash="dash", color=COLOR_BEST),
        showlegend=False
    ), row=row_rmse, col=2)

    fig.add_trace(go.Scatter(
        x=H, y=res["rmse"][cat_name]["cv"],
        name="Constant velocity", mode="lines+markers",
        line=dict(width=2, dash="dot", color=COLOR_CV),
        showlegend=False
    ), row=row_rmse, col=2)

# Add traces — legend only once (for first row)
add_metric_traces("3d",   1, 1, show_legend=True)
add_metric_traces("2d",   2, 2, show_legend=False)
add_metric_traces("vert", 3, 3, show_legend=False)

# --- layout ---
fig.update_layout(
    template="plotly_white",
    height=900, width=1100,
    # title="MAE and RMSE vs Prediction Horizon",
    legend=dict(
        orientation="h",
        yanchor="bottom", y=-0.1,
        xanchor="center", x=0.5,
    )
)

# axis labels
for r in range(1, 4):
    fig.update_xaxes(title_text="Horizon [s]", row=r, col=1)
    fig.update_xaxes(title_text="Horizon [s]", row=r, col=2)
    fig.update_yaxes(title_text="Error [m]", row=r, col=1)
    fig.update_yaxes(title_text="Error [m]", row=r, col=2)

fig.show()

## Probabilistic Calibration Analysis

Analyze the calibration of probabilistic predictions using PIT histograms.

In [53]:
@torch.no_grad()
def collect_samples_for_calibration(
    model,
    X,
    Y,
    C,
    feat_mean,
    feat_std,
    ctx_mean,
    ctx_std,
    device,
    preprocess=None,
    n_eval: int = 2000,
    n_samples: int = 64,
    n_steps: int = 64,
    batch_size: int = 128,
):
    """Draw samples and return predictions and ground truth."""
    N = min(int(X.shape[0]), int(n_eval))
    T = int(Y.shape[1])
    pos_samples = []
    pos_truth = []
    for i0 in tqdm(range(0, N, batch_size)):
        i1 = min(N, i0 + batch_size)
        x_hist = torch.from_numpy(np.array(X[i0:i1])).to(device).contiguous()
        ctx = torch.from_numpy(np.array(C[i0:i1])).to(device).contiguous()
        y_true = torch.from_numpy(np.array(Y[i0:i1])).to(device).contiguous()
        B = int(x_hist.shape[0])
        ctx_model = _ctx_from_intent(x_hist, ctx, preprocess)
        y_norm_all = sample_many(
            model,
            x_hist,
            ctx_model,
            T_out=T,
            n_steps=n_steps,
            G=1.0,
            n_samples=n_samples,
            chunk=128,
        )  # (S*B, T, Dn)
        y_glob_all = denorm_seq_to_global(
            y_norm_all, ctx.repeat(n_samples, 1), feat_mean, feat_std, ctx_mean, ctx_std
        ).view(n_samples, B, T, -1)[..., :3]  # (S,B,T,3)
        y_true_g = denorm_seq_to_global(
            y_true, ctx, feat_mean, feat_std, ctx_mean, ctx_std
        )[..., :3]
        pos_samples.append(y_glob_all)
        pos_truth.append(y_true_g)
    ysamp_g = torch.cat(pos_samples, dim=1)  # (S,N,T,3)
    ytrue_g = torch.cat(pos_truth, dim=0)  # (N,T,3)
    return ysamp_g, ytrue_g

# Draw samples for calibration analysis
ysamp_g, ytrue_g = collect_samples_for_calibration(
    model,
    X_test,
    Y_test,
    C_test,
    feat_mean,
    feat_std,
    ctx_mean,
    ctx_std,
    device=DEVICE,
    preprocess=CKPT_PREPROCESS,
    n_eval=12,
    n_samples=256,
    n_steps=32,
    batch_size=4,
)

print(f"Collected {ysamp_g.shape[0]} samples for {ysamp_g.shape[1]} trajectories")

  0%|          | 0/3 [00:00<?, ?it/s]

Collected 256 samples for 12 trajectories


## PIT Histogram Visualization

Create Probability Integral Transform (PIT) histograms to assess calibration.

In [54]:
# Calculate PIT values
pits_btd = pit_values(ysamp_g, ytrue_g)  # (N,T,3)
pits_flat = pits_btd.reshape(-1, 3).detach().cpu().numpy()

# Setup for plotting
axis_names = ["x", "y", "z"]
axis_colors = ["black", "red", "blue"]
uniform_line_color = "gray"

# Create subplots: 1 row x 3 cols (histograms)
fig = make_subplots(
    rows=1, cols=3,
    shared_xaxes=False,
    shared_yaxes=False,
    horizontal_spacing=0.07,
    subplot_titles=[f"PIT histogram — {axis}" for axis in axis_names],
)

# Parameters
nbins = 30

for d in range(3):
    col = d + 1
    color = axis_colors[d]
    u = pits_flat[:, d]

    # Histogram
    fig.add_trace(
        go.Histogram(
            x=u,
            nbinsx=nbins,
            histnorm="probability density",
            marker=dict(color=color),
            name=axis_names[d],
            showlegend=False,
            opacity=0.75,
        ),
        row=1, col=col
    )

    # Uniform PDF reference line (y=1 on [0,1])
    fig.add_trace(
        go.Scatter(
            x=[0, 1],
            y=[1, 1],
            mode="lines",
            line=dict(dash="dash", color=uniform_line_color),
            name="Uniform PDF (y=1)" if d == 0 else None,
            showlegend=(d == 0),
        ),
        row=1, col=col
    )

    # Axes formatting
    fig.update_xaxes(title_text="PIT value", range=[0, 1], row=1, col=col)
    fig.update_yaxes(title_text="Density", range=[0, 4.5], row=1, col=col)

# Layout
fig.update_layout(
    height=400, width=800,
    template="plotly_white",
    title="Calibration diagnostics: PIT histograms",
    bargap=0.05,
    legend=dict(
        orientation="h",
        yanchor="bottom", y=-0.3,
        xanchor="center", x=0.5
    ),
    margin=dict(l=10, r=10, t=60, b=60),
)

# Save and display
#fig.write_html("../figures/pit_histograms.html")
fig.show()

In [ ]:
import pandas as pd
from utils.metrics import ade_fde, pit_values, coverage_curve_1d, positional_spread, crps_positions, energy_score_per_horizon

# ============================================================================
# COMPREHENSIVE EVALUATION SWEEP: epoch40 vs epoch52
# ============================================================================

print("=" * 80)
print("COMPREHENSIVE EVALUATION SWEEP: cfm_base_epoch40.pt vs cfm_base.pt (epoch52)")
print("=" * 80)

# Configuration
EVAL_SUBSET_SIZE = 120  # balanced for statistics and compute
EVAL_SAMPLES_PER_CASE = 32  # standard sampling for metrics
EVAL_N_STEPS = 64  # full resolution sampling
RNG_SEED = 42

# Select deterministic subset
rng_eval = np.random.default_rng(RNG_SEED)
eval_cases = rng_eval.choice(len(X_test), size=EVAL_SUBSET_SIZE, replace=False).tolist()

# Prepare test data
x_eval = torch.from_numpy(np.asarray(X_test[eval_cases])).to(DEVICE).contiguous()
ctx_eval = torch.from_numpy(np.asarray(C_test[eval_cases])).to(DEVICE).contiguous()
y_eval = torch.from_numpy(np.asarray(Y_test[eval_cases])).to(DEVICE).contiguous()

print(f"\n[Setup] Evaluation subset: {len(eval_cases)} cases × {EVAL_SAMPLES_PER_CASE} samples")
print(f"[Setup] Sampling: n_steps={EVAL_N_STEPS}, dt={OUTPUT_STRIDE_SECONDS}s")

# ============================================================================
# Load both checkpoints
# ============================================================================

# Checkpoint 1: epoch 40
ckpt_epoch40 = "../models/cfm_base_epoch40.pt"
model_epoch40 = load_base_checkpoint(ckpt_epoch40, device=DEVICE, strict=False)
model_epoch40.eval()
print(f"\n✓ Loaded cfm_base_epoch40.pt")

# Checkpoint 2: epoch 52 (current model already loaded, or reload)
model_epoch52 = model  # Already loaded in kernel
print(f"✓ Using cfm_base.pt (epoch52) from kernel")

# ============================================================================
# Sample from both models
# ============================================================================

print(f"\n[Sampling] Generating {EVAL_SAMPLES_PER_CASE} samples per case from both models...")

with torch.no_grad():
    # Epoch 40
    y_samp_epoch40_norm = sample_many(
        model_epoch40,
        x_eval,
        ctx_eval,
        T_out=int(Y_test.shape[1]),
        n_steps=EVAL_N_STEPS,
        G=1.0,
        n_samples=EVAL_SAMPLES_PER_CASE,
        chunk=4,
    ).reshape(EVAL_SAMPLES_PER_CASE, EVAL_SUBSET_SIZE, int(Y_test.shape[1]), -1)

    # Epoch 52
    y_samp_epoch52_norm = sample_many(
        model_epoch52,
        x_eval,
        ctx_eval,
        T_out=int(Y_test.shape[1]),
        n_steps=EVAL_N_STEPS,
        G=1.0,
        n_samples=EVAL_SAMPLES_PER_CASE,
        chunk=4,
    ).reshape(EVAL_SAMPLES_PER_CASE, EVAL_SUBSET_SIZE, int(Y_test.shape[1]), -1)

# Denormalize to global frame
y_samp_epoch40_glob = denorm_seq_to_global(
    y_samp_epoch40_norm.reshape(-1, int(Y_test.shape[1]), y_samp_epoch40_norm.shape[-1]),
    ctx_eval.repeat(EVAL_SAMPLES_PER_CASE, 1),
    feat_mean,
    feat_std,
    ctx_mean,
    ctx_std,
).reshape(EVAL_SAMPLES_PER_CASE, EVAL_SUBSET_SIZE, int(Y_test.shape[1]), -1)

y_samp_epoch52_glob = denorm_seq_to_global(
    y_samp_epoch52_norm.reshape(-1, int(Y_test.shape[1]), y_samp_epoch52_norm.shape[-1]),
    ctx_eval.repeat(EVAL_SAMPLES_PER_CASE, 1),
    feat_mean,
    feat_std,
    ctx_mean,
    ctx_std,
).reshape(EVAL_SAMPLES_PER_CASE, EVAL_SUBSET_SIZE, int(Y_test.shape[1]), -1)

y_eval_glob = denorm_seq_to_global(
    y_eval,
    ctx_eval,
    feat_mean,
    feat_std,
    ctx_mean,
    ctx_std,
)

print(f"✓ Sampling complete")

# ============================================================================
# COMPUTE METRICS
# ============================================================================

metrics_results = {}

print(f"\n[Metrics] Computing...\n")

# 1. ADE / FDE (Average/Final Displacement Error)
ade_e40, fde_e40 = ade_fde(y_samp_epoch40_glob.mean(dim=0), y_eval_glob)
ade_e52, fde_e52 = ade_fde(y_samp_epoch52_glob.mean(dim=0), y_eval_glob)
metrics_results['ade_fde'] = {
    'epoch40': {'ade_mean': float(ade_e40.mean()), 'ade_std': float(ade_e40.std()), 'fde_mean': float(fde_e40.mean()), 'fde_std': float(fde_e40.std())},
    'epoch52': {'ade_mean': float(ade_e52.mean()), 'ade_std': float(ade_e52.std()), 'fde_mean': float(fde_e52.mean()), 'fde_std': float(fde_e52.std())},
}
print(f"✓ ADE/FDE:")
print(f"  epoch40: ADE={metrics_results['ade_fde']['epoch40']['ade_mean']:.2f}m ± {metrics_results['ade_fde']['epoch40']['ade_std']:.2f}, FDE={metrics_results['ade_fde']['epoch40']['fde_mean']:.2f}m")
print(f"  epoch52: ADE={metrics_results['ade_fde']['epoch52']['ade_mean']:.2f}m ± {metrics_results['ade_fde']['epoch52']['ade_std']:.2f}, FDE={metrics_results['ade_fde']['epoch52']['fde_mean']:.2f}m")

# 2. PIT (Probability Integral Transform) - calibration
pit_e40 = pit_values(y_samp_epoch40_glob[..., :3], y_eval_glob[..., :3])
pit_e52 = pit_values(y_samp_epoch52_glob[..., :3], y_eval_glob[..., :3])
# PIT should be uniform (0.5 mean)
pit_e40_mean = float(pit_e40.mean())
pit_e52_mean = float(pit_e52.mean())
metrics_results['pit'] = {
    'epoch40': {'mean': pit_e40_mean, 'std': float(pit_e40.std())},
    'epoch52': {'mean': pit_e52_mean, 'std': float(pit_e52.std())},
}
print(f"\n✓ PIT (calibration, ideal=0.50):")
print(f"  epoch40: mean={pit_e40_mean:.4f} ± {metrics_results['pit']['epoch40']['std']:.4f}")
print(f"  epoch52: mean={pit_e52_mean:.4f} ± {metrics_results['pit']['epoch52']['std']:.4f}")

# 3. Coverage (empirical coverage for central 50%, 90% intervals)
cov_e40 = coverage_curve_1d(y_samp_epoch40_glob[..., :3], y_eval_glob[..., :3], alphas=[0.5, 0.9])
cov_e52 = coverage_curve_1d(y_samp_epoch52_glob[..., :3], y_eval_glob[..., :3], alphas=[0.5, 0.9])
metrics_results['coverage'] = {
    'epoch40': {'50%': float(cov_e40[0.5].mean()), '90%': float(cov_e40[0.9].mean())},
    'epoch52': {'50%': float(cov_e52[0.5].mean()), '90%': float(cov_e52[0.9].mean())},
}
print(f"\n✓ Coverage (empirical vs nominal):")
print(f"  epoch40: 50% coverage={metrics_results['coverage']['epoch40']['50%']:.4f} (ideal=0.50), 90% coverage={metrics_results['coverage']['epoch40']['90%']:.4f} (ideal=0.90)")
print(f"  epoch52: 50% coverage={metrics_results['coverage']['epoch52']['50%']:.4f} (ideal=0.50), 90% coverage={metrics_results['coverage']['epoch52']['90%']:.4f} (ideal=0.90)")

# 4. Spread / RMSE (variance of samples at each horizon)
spread_e40 = positional_spread(y_samp_epoch40_glob[..., :3])
spread_e52 = positional_spread(y_samp_epoch52_glob[..., :3])
metrics_results['spread'] = {
    'epoch40': {'mean': float(spread_e40.mean()), 'std': float(spread_e40.std()), 'median': float(spread_e40.median())},
    'epoch52': {'mean': float(spread_e52.mean()), 'std': float(spread_e52.std()), 'median': float(spread_e52.median())},
}
print(f"\n✓ Spread/RMSE (sample variance):")
print(f"  epoch40: mean={metrics_results['spread']['epoch40']['mean']:.2f}m ± {metrics_results['spread']['epoch40']['std']:.2f}m")
print(f"  epoch52: mean={metrics_results['spread']['epoch52']['mean']:.2f}m ± {metrics_results['spread']['epoch52']['std']:.2f}m")

# 5. CRPS / Energy Score (calibration proper scoring rule)
crps_e40 = crps_positions(y_samp_epoch40_glob, y_eval_glob)
crps_e52 = crps_positions(y_samp_epoch52_glob, y_eval_glob)
energy_e40 = energy_score_per_horizon(y_samp_epoch40_glob[..., :3], y_eval_glob[..., :3])
energy_e52 = energy_score_per_horizon(y_samp_epoch52_glob[..., :3], y_eval_glob[..., :3])
metrics_results['crps_energy'] = {
    'epoch40': {'crps_mean': float(crps_e40.mean()), 'crps_std': float(crps_e40.std()), 'energy_mean': float(energy_e40.mean()), 'energy_std': float(energy_e40.std())},
    'epoch52': {'crps_mean': float(crps_e52.mean()), 'crps_std': float(crps_e52.std()), 'energy_mean': float(energy_e52.mean()), 'energy_std': float(energy_e52.std())},
}
print(f"\n✓ CRPS (lower is better):")
print(f"  epoch40: mean={metrics_results['crps_energy']['epoch40']['crps_mean']:.4f} ± {metrics_results['crps_energy']['epoch40']['crps_std']:.4f}")
print(f"  epoch52: mean={metrics_results['crps_energy']['epoch52']['crps_mean']:.4f} ± {metrics_results['crps_energy']['epoch52']['crps_std']:.4f}")
print(f"\n✓ Energy Score (lower is better):")
print(f"  epoch40: mean={metrics_results['crps_energy']['epoch40']['energy_mean']:.4f} ± {metrics_results['crps_energy']['epoch40']['energy_std']:.4f}")
print(f"  epoch52: mean={metrics_results['crps_energy']['epoch52']['energy_mean']:.4f} ± {metrics_results['crps_energy']['epoch52']['energy_std']:.4f}")

# 6. Physics Residual (kinematic consistency check)
print(f"\n[Physics] Computing kinematic residuals...")

# Epoch 40
pred_pos_e40 = y_samp_epoch40_glob[..., :3]
pred_vel_e40 = y_samp_epoch40_glob[..., 3:6]
pred_dpos_e40 = pred_pos_e40[:, :, 1:, :] - pred_pos_e40[:, :, :-1, :]
pred_expected_e40 = pred_vel_e40[:, :, :-1, :] * float(OUTPUT_STRIDE_SECONDS)
pred_residual_e40 = pred_dpos_e40 - pred_expected_e40
pred_residual_norm_e40 = np.linalg.norm(pred_residual_e40.cpu().numpy(), axis=-1)

# Epoch 52
pred_pos_e52 = y_samp_epoch52_glob[..., :3]
pred_vel_e52 = y_samp_epoch52_glob[..., 3:6]
pred_dpos_e52 = pred_pos_e52[:, :, 1:, :] - pred_pos_e52[:, :, :-1, :]
pred_expected_e52 = pred_vel_e52[:, :, :-1, :] * float(OUTPUT_STRIDE_SECONDS)
pred_residual_e52 = pred_dpos_e52 - pred_expected_e52
pred_residual_norm_e52 = np.linalg.norm(pred_residual_e52.cpu().numpy(), axis=-1)

# Ground truth (for reference)
gt_pos = y_eval_glob[..., :3]
gt_vel = y_eval_glob[..., 3:6]
gt_dpos = gt_pos[:, 1:, :] - gt_pos[:, :-1, :]
gt_expected = gt_vel[:, :-1, :] * float(OUTPUT_STRIDE_SECONDS)
gt_residual = (gt_dpos - gt_expected).cpu().numpy()
gt_residual_norm = np.linalg.norm(gt_residual, axis=-1)

metrics_results['physics_residual'] = {
    'epoch40': {'mean_m': float(np.mean(pred_residual_norm_e40)), 'median_m': float(np.median(pred_residual_norm_e40)), 'std_m': float(np.std(pred_residual_norm_e40))},
    'epoch52': {'mean_m': float(np.mean(pred_residual_norm_e52)), 'median_m': float(np.median(pred_residual_norm_e52)), 'std_m': float(np.std(pred_residual_norm_e52))},
    'ground_truth': {'mean_m': float(np.mean(gt_residual_norm)), 'median_m': float(np.median(gt_residual_norm)), 'std_m': float(np.std(gt_residual_norm))},
}
print(f"\n✓ Physics Residual (position - velocity × dt):")
print(f"  epoch40: mean={metrics_results['physics_residual']['epoch40']['mean_m']:.2f}m ± {metrics_results['physics_residual']['epoch40']['std_m']:.2f}m")
print(f"  epoch52: mean={metrics_results['physics_residual']['epoch52']['mean_m']:.2f}m ± {metrics_results['physics_residual']['epoch52']['std_m']:.2f}m")
print(f"  ground truth (reference): mean={metrics_results['physics_residual']['ground_truth']['mean_m']:.2f}m ± {metrics_results['physics_residual']['ground_truth']['std_m']:.2f}m")
print(f"  residual ratio (epoch40/gt): {metrics_results['physics_residual']['epoch40']['mean_m'] / metrics_results['physics_residual']['ground_truth']['mean_m']:.1f}x")
print(f"  residual ratio (epoch52/gt): {metrics_results['physics_residual']['epoch52']['mean_m'] / metrics_results['physics_residual']['ground_truth']['mean_m']:.1f}x")

# ============================================================================
# SUMMARY TABLE & COMPARISON
# ============================================================================

print("\n" + "=" * 80)
print("COMPARATIVE SUMMARY")
print("=" * 80)

summary_df = pd.DataFrame({
    'Metric': [
        'ADE (m)',
        'FDE (m)',
        'PIT (ideal=0.50)',
        'Coverage 50% (ideal=0.50)',
        'Coverage 90% (ideal=0.90)',
        'Spread (m)',
        'CRPS',
        'Energy Score',
        'Physics Residual (m)',
    ],
    'epoch40': [
        f"{metrics_results['ade_fde']['epoch40']['ade_mean']:.2f}",
        f"{metrics_results['ade_fde']['epoch40']['fde_mean']:.2f}",
        f"{metrics_results['pit']['epoch40']['mean']:.4f}",
        f"{metrics_results['coverage']['epoch40']['50%']:.4f}",
        f"{metrics_results['coverage']['epoch40']['90%']:.4f}",
        f"{metrics_results['spread']['epoch40']['mean']:.2f}",
        f"{metrics_results['crps_energy']['epoch40']['crps_mean']:.4f}",
        f"{metrics_results['crps_energy']['epoch40']['energy_mean']:.4f}",
        f"{metrics_results['physics_residual']['epoch40']['mean_m']:.2f}",
    ],
    'epoch52': [
        f"{metrics_results['ade_fde']['epoch52']['ade_mean']:.2f}",
        f"{metrics_results['ade_fde']['epoch52']['fde_mean']:.2f}",
        f"{metrics_results['pit']['epoch52']['mean']:.4f}",
        f"{metrics_results['coverage']['epoch52']['50%']:.4f}",
        f"{metrics_results['coverage']['epoch52']['90%']:.4f}",
        f"{metrics_results['spread']['epoch52']['mean']:.2f}",
        f"{metrics_results['crps_energy']['epoch52']['crps_mean']:.4f}",
        f"{metrics_results['crps_energy']['epoch52']['energy_mean']:.4f}",
        f"{metrics_results['physics_residual']['epoch52']['mean_m']:.2f}",
    ],
})

# Calculate improvements
def calc_improvement(v_old, v_new, lower_is_better=True):
    v_old, v_new = float(v_old), float(v_new)
    pct = ((v_old - v_new) / (abs(v_old) + 1e-6)) * 100
    if lower_is_better:
        return f"{pct:+.1f}%" if pct != 0 else "→"
    else:
        return f"{pct:+.1f}%" if pct != 0 else "→"

summary_df['Δ (40→52)'] = [
    calc_improvement(metrics_results['ade_fde']['epoch40']['ade_mean'], metrics_results['ade_fde']['epoch52']['ade_mean'], lower_is_better=True),
    calc_improvement(metrics_results['ade_fde']['epoch40']['fde_mean'], metrics_results['ade_fde']['epoch52']['fde_mean'], lower_is_better=True),
    calc_improvement(abs(metrics_results['pit']['epoch40']['mean'] - 0.5), abs(metrics_results['pit']['epoch52']['mean'] - 0.5), lower_is_better=True),
    calc_improvement(abs(metrics_results['coverage']['epoch40']['50%'] - 0.5), abs(metrics_results['coverage']['epoch52']['50%'] - 0.5), lower_is_better=True),
    calc_improvement(abs(metrics_results['coverage']['epoch40']['90%'] - 0.9), abs(metrics_results['coverage']['epoch52']['90%'] - 0.9), lower_is_better=True),
    calc_improvement(metrics_results['spread']['epoch40']['mean'], metrics_results['spread']['epoch52']['mean'], lower_is_better=False),  # spread: doesn't have clear direction
    calc_improvement(metrics_results['crps_energy']['epoch40']['crps_mean'], metrics_results['crps_energy']['epoch52']['crps_mean'], lower_is_better=True),
    calc_improvement(metrics_results['crps_energy']['epoch40']['energy_mean'], metrics_results['crps_energy']['epoch52']['energy_mean'], lower_is_better=True),
    calc_improvement(metrics_results['physics_residual']['epoch40']['mean_m'], metrics_results['physics_residual']['epoch52']['mean_m'], lower_is_better=True),
]

print(summary_df.to_string(index=False))

# ============================================================================
# DECISION GUIDANCE
# ============================================================================

print("\n" + "=" * 80)
print("DECISION GATE ANALYSIS")
print("=" * 80)

improvements = {
    'ade': float(metrics_results['ade_fde']['epoch40']['ade_mean']) > float(metrics_results['ade_fde']['epoch52']['ade_mean']),
    'fde': float(metrics_results['ade_fde']['epoch40']['fde_mean']) > float(metrics_results['ade_fde']['epoch52']['fde_mean']),
    'pit': abs(metrics_results['pit']['epoch40']['mean'] - 0.5) > abs(metrics_results['pit']['epoch52']['mean'] - 0.5),
    'coverage_50': abs(metrics_results['coverage']['epoch40']['50%'] - 0.5) > abs(metrics_results['coverage']['epoch52']['50%'] - 0.5),
    'coverage_90': abs(metrics_results['coverage']['epoch40']['90%'] - 0.9) > abs(metrics_results['coverage']['epoch52']['90%'] - 0.9),
    'crps': float(metrics_results['crps_energy']['epoch40']['crps_mean']) > float(metrics_results['crps_energy']['epoch52']['crps_mean']),
    'energy': float(metrics_results['crps_energy']['epoch40']['energy_mean']) > float(metrics_results['crps_energy']['epoch52']['energy_mean']),
    'physics': float(metrics_results['physics_residual']['epoch40']['mean_m']) > float(metrics_results['physics_residual']['epoch52']['mean_m']),
}

n_improved = sum(improvements.values())
n_total = len(improvements)

print(f"\nMetrics improved epoch40→epoch52: {n_improved}/{n_total}")
for metric, improved in improvements.items():
    status = "✓" if improved else "✗"
    print(f"  {status} {metric}")

print("\n" + "-" * 80)
print("RECOMMENDATIONS:")
print("-" * 80)

if n_improved >= 6:
    print("\n✓ TRAINING HELPS: Epoch 52 shows consistent improvements across metrics.")
    print("  → Continue training longer (e.g., epoch 60+) to see if gains continue.")
    print("  → Physics residual may improve with more training without explicit loss term.")
elif n_improved >= 4:
    print("\n⚠ MIXED RESULTS: Some metrics improved, others unchanged/regressed.")
    print("  → Training provides partial benefit.")
    print("  → Physics residual suggests physics loss term may still be needed.")
    print("  → Consider: (a) continue training, or (b) add physics loss term.")
else:
    print("\n✗ TRAINING NOT SUFFICIENT: Few improvements from epoch 40→52.")
    print("  → Continued training alone may not solve kinematic consistency issue.")
    print("  → RECOMMENDATION: Implement physics loss term (kin_w > 0) in trainer.")
    print("  → Re-train with physics loss from epoch 40 or earlier checkpoint.")

print(f"\nPhysics residual analysis:")
phys_ratio_40 = metrics_results['physics_residual']['epoch40']['mean_m'] / metrics_results['physics_residual']['ground_truth']['mean_m']
phys_ratio_52 = metrics_results['physics_residual']['epoch52']['mean_m'] / metrics_results['physics_residual']['ground_truth']['mean_m']
print(f"  Epoch 40: {phys_ratio_40:.1f}x ground truth residual")
print(f"  Epoch 52: {phys_ratio_52:.1f}x ground truth residual")
print(f"  Improvement: {(phys_ratio_40 - phys_ratio_52) / phys_ratio_40 * 100:.1f}%")

if phys_ratio_52 > 10:
    print(f"  → Physics residual still very high ({phys_ratio_52:.1f}x). Explicit loss term recommended.")
elif phys_ratio_52 > 5:
    print(f"  → Physics residual improved but still significant. Consider physics loss term.")
else:
    print(f"  → Physics residual acceptable. Current training approach may be sufficient.")

print("\n" + "=" * 80)